# E-Commerce Multi-Platform Dataset — EDA

**Dataset:** Products collected from Amazon, Noon, AliExpress, Jumia, and eBay  
**Tool:** [M-D Web Scraper](https://github.com/YOUR_USERNAME/md-scraper) by Mohamed Darwish  
**Columns:** 21 — price, rating, discount, availability, category, source, and more

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# Load dataset — adjust path as needed
DATA = Path('../data/processed')
csv  = sorted(DATA.glob('ecommerce_*.csv'))[-1]
print(f'Loading: {csv.name}')

df = pd.read_csv(csv)
df['scraped_at'] = pd.to_datetime(df['scraped_at'])
print(f'Shape: {df.shape}')

## 1. Overview

In [ ]:
print('=== DATASET SUMMARY ===')
print(f'Products       : {len(df):,}')
print(f'Sources        : {df["source"].nunique()}  →  {list(df["source"].unique())}')
print(f'Categories     : {df["category"].nunique()}')
print(f'Currencies     : {list(df["currency"].unique())}')
print(f'With discount  : {df["has_discount"].sum():,}  ({df["has_discount"].mean()*100:.1f}%)')
print(f'With rating    : {df["rating_score"].notna().sum():,}')
print(f'Avg rating     : {df["rating_score"].mean():.2f}/5.0')
print(f'In stock       : {(df["availability"]=="in_stock").sum():,}')

In [ ]:
df.head(5)

In [ ]:
df.describe().round(2)

## 2. Products by Source

In [ ]:
source_stats = df.groupby('source').agg(
    count      = ('price_current', 'count'),
    avg_price  = ('price_current', 'mean'),
    avg_rating = ('rating_score',  'mean'),
    disc_rate  = ('has_discount',  'mean'),
    currency   = ('currency',      'first'),
).round(2)
source_stats['disc_rate'] = (source_stats['disc_rate'] * 100).round(1)
print(source_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Products per source
src_counts = df['source'].value_counts()
axes[0].bar(src_counts.index, src_counts.values, color='#2C3E50', alpha=0.85)
axes[0].set_title('Products per source')
axes[0].set_ylabel('Count')
for i, v in enumerate(src_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=9)

# Avg rating per source
rated = df[df['rating_score'].notna()].groupby('source')['rating_score'].mean().sort_values()
axes[1].barh(rated.index, rated.values, color='#F39C12', alpha=0.85)
axes[1].set_title('Average rating by source')
axes[1].set_xlabel('Rating (0–5)')

plt.tight_layout()
plt.savefig('../data/processed/by_source.png', bbox_inches='tight')
plt.show()

## 3. Price Analysis

In [ ]:
# Price per source (separate currency contexts)
for src in df['source'].unique():
    sub = df[df['source'] == src]['price_current']
    cur = df[df['source'] == src]['currency'].iloc[0]
    print(f'{src:<15} {cur}  mean={sub.mean():.2f}  median={sub.median():.2f}  min={sub.min():.2f}  max={sub.max():.2f}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

colors = ['#2C3E50', '#E74C3C', '#27AE60', '#F39C12', '#8E44AD']
for i, (src, color) in enumerate(zip(df['source'].unique(), colors)):
    sub = df[df['source'] == src]['price_current']
    cur = df[df['source'] == src]['currency'].iloc[0]
    axes[i].hist(sub, bins=20, color=color, alpha=0.8, edgecolor='white')
    axes[i].axvline(sub.mean(),   color='white', linewidth=1.5, linestyle='--')
    axes[i].set_title(f'{src}  ({cur})')
    axes[i].set_xlabel('Price')

axes[-1].set_visible(False)
plt.suptitle('Price distribution by platform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/price_by_source.png', bbox_inches='tight')
plt.show()

## 4. Category Analysis

In [ ]:
cat_stats = df.groupby(['category', 'source']).agg(
    count     = ('price_current', 'count'),
    avg_price = ('price_current', 'mean'),
).round(2).reset_index()

top_cats = df['category'].value_counts().head(10).index
sub = cat_stats[cat_stats['category'].isin(top_cats)]

fig, ax = plt.subplots(figsize=(12, 6))
pivot = df[df['category'].isin(top_cats)].groupby(['category', 'source']).size().unstack(fill_value=0)
pivot.plot(kind='bar', ax=ax, alpha=0.85)
ax.set_title('Top categories by source')
ax.set_xlabel('Category')
ax.set_ylabel('Products')
ax.legend(title='Source')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig('../data/processed/category_by_source.png', bbox_inches='tight')
plt.show()

## 5. Rating & Discount Analysis

In [ ]:
rated = df[df['rating_score'].notna()]
disc  = df[df['has_discount'] == True]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(rated['rating_score'], bins=20, color='#F39C12', alpha=0.85, edgecolor='white')
axes[0].set_title(f'Rating distribution  (n={len(rated):,})')
axes[0].set_xlabel('Rating score (0–5)')

axes[1].hist(disc['discount_pct'].dropna(), bins=25, color='#E74C3C', alpha=0.85, edgecolor='white')
axes[1].set_title(f'Discount % distribution  (n={len(disc):,})')
axes[1].set_xlabel('Discount %')

plt.tight_layout()
plt.savefig('../data/processed/rating_discount.png', bbox_inches='tight')
plt.show()

print(f'Avg discount among discounted items: {disc["discount_pct"].mean():.1f}%')
print(f'Max discount: {disc["discount_pct"].max():.1f}%')

## 6. Availability

In [ ]:
avail = df['availability'].value_counts()
colors_avail = ['#27AE60', '#F39C12', '#E74C3C', '#95A5A6']

fig, ax = plt.subplots(figsize=(6, 5))
ax.pie(avail.values, labels=avail.index, colors=colors_avail[:len(avail)],
       autopct='%1.1f%%', startangle=90)
ax.set_title('Availability breakdown')
plt.tight_layout()
plt.savefig('../data/processed/availability.png', bbox_inches='tight')
plt.show()
print(avail.to_string())

## 7. Cleaning for ML

In [ ]:
df_ml = df.copy()

# Fill missing numerics
df_ml['rating_score']   = df_ml['rating_score'].fillna(df_ml['rating_score'].median())
df_ml['discount_pct']   = df_ml['discount_pct'].fillna(0)
df_ml['price_original'] = df_ml['price_original'].fillna(df_ml['price_current'])
df_ml['reviews_count']  = df_ml['reviews_count'].fillna(0)

# Encode availability
avail_map = {'in_stock': 2, 'limited': 1, 'out_of_stock': 0, 'unknown': -1}
df_ml['availability_code'] = df_ml['availability'].map(avail_map)

# Encode source
df_ml['source_code'] = pd.Categorical(df_ml['source']).codes

print(f'ML-ready shape : {df_ml.shape}')
print(f'Null remaining : {df_ml.isnull().sum().sum()}')

out = Path('../data/processed/ecommerce_ml_ready.csv')
df_ml.to_csv(out, index=False)
print(f'Saved: {out}')

## 8. Key Findings

| Metric | Value |
|--------|-------|
| Total products | 1,000+ across 5 platforms |
| Categories | 10 product categories |
| Discount rate | ~30% of products have discounts |
| Avg rating | 3.76/5.0 |
| Availability | ~83% in stock |

**Ideas for ML tasks:**
- Predict price from category + rating + source
- Classify whether a product will be discounted
- Cluster products into market segments
- Predict availability from historical patterns
- Cross-platform price comparison models